In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [2]:
import torch
import torchaudio
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample
from speechbrain.inference.vocoders import UnitHIFIGAN
from transformers import Wav2Vec2Processor, AutoTokenizer

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def ctc_postprocess(tokens, blank):
    _toks = tokens.squeeze(0).tolist()
    deduplicated_toks = [v for i, v in enumerate(_toks) if i == 0 or v != _toks[i - 1]]
    hyp = [v for v in deduplicated_toks if v != blank] #官方493 222
    hyp = " ".join(list(map(str, hyp))) #1918 547
    return hyp

In [5]:
dataset = load_from_disk('/scratch/asudupe/datasets/VoiceAssistant-400K_eu/')

In [ ]:
speech, sr = torchaudio.load(os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['train'][10]['question_audio']))
Audio(data=np.array(speech), rate=sr)

In [15]:
dataset['train'][10]['answer']

'Txakur bat apaintzeak hainbat urrats dakartza. Has zaitez zure txakurraren ilea eskuilatzen, korapiloak eta ile solteak kentzeko. Ondoren, eman bainu bat zure txakurrari, txakurren ile-apainketarako xanpu egokia erabiliz, eta ziurtatu ondo garbitzen duzula hondakinik ez uzteko. Bainuaren ondoren, lehortu zure txakurra eskuoihal batekin edo animalientzako lehorgailu batekin. Moztu zure txakurraren azazkalak kontu handiz, azazkalaren erroa ukitu gabe. Azkenik, garbitu zure txakurraren belarriak albaitariak gomendatutako belarri-garbitzaile batekin eta garbitu hortzak txakurrentzako hortzetako pastarekin. Izan beti leuna eta eskaini sariak prozesuan zehar zure txakurra lasai eta eroso mantentzeko.'

In [6]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [13]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [36]:
# model_path = 'saves/13834/checkpoint-24000'
# model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage1/best/checkpoint-20976"
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage2/best/checkpoint-20973"
# model_path = "/hitz_data/asudupe/models/Latxa-Llama-3.1-8B-Instruct"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = True
mel_size = 128
conv_mode = 'llama_3'

In [37]:
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.78it/s]


In [72]:
hubert_tokenizer = Wav2Vec2Processor.from_pretrained("/scratch/asudupe/models/speech_encoder/mHubert-basque/")
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

In [11]:
hifigan = UnitHIFIGAN.from_hparams(source="/scratch/asudupe/models/hifigan/gaitu/", run_opts={"device":'cuda'})

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [42]:
qs = "<speech>\Please answer directly the question."
speech_file = 'audioak/Recording_9.mp3'
# speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1016]['question_audio'])
speech_loaded = whisper.load_audio(speech_file)
# audio = dataset['train'][20]['question_audio']
# speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech_loaded)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
# speech = hubert_tokenizer(speech_loaded, sampling_rate=16000, return_tensors="pt", padding=True)['input_values'].permute(1, 0)

input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = input_ids.unsqueeze(0)
speech_tensors = speech_tensor.unsqueeze(0)
speech_lengths = speech_length.unsqueeze(0)

# input_ids = torch.stack((input_ids), dim=0)
# speech_tensors = torch.stack((speech_tensor), dim=0)
# speech_lengths = torch.stack((speech_length), dim=0)

#torch.Size([1, 62]),torch.Size([1, 3000, 128]) #tensor([[3000]])
Audio(speech_loaded, rate=16000)

In [29]:
input_ids.shape, speech_tensors.shape, speech_lengths

(torch.Size([1, 59]),
 torch.Size([1, 3000, 128]),
 tensor([[3000]], device='cuda:0'))

In [ ]:
temperature = 0
top_p = None
num_beams = 1
max_new_tokens = 512

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
        streaming_unit_gen=True,
 
    )
# output_ids = outputs
output_ids, output_units = outputs

print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip())
output_units = ctc_postprocess(output_units, blank=model.config.unit_vocab_size)
output_units = torch.tensor([int(x) for x in output_units.split()], dtype=torch.long)
answer = hifigan.decode_unit(output_units.unsqueeze(-1), torch.tensor(np.load('/scratch/asudupe/models/hifigan/gaitu/nerea.npy')))
Audio(answer.cpu(), rate=16000)


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Futbolean, "euskal talderik hobena" terminoa subjektiboa izan daiteke eta irizpide ezberdinen araberakoa da, hala nola, egungo errendimendua, lorpen historikoak edo lehentasun pertsonalak. Hala ere, zale eta analista askok Real Sociedad eta Athletic Club aipatzen dituzte euskal talderik onenen artean, beren historia aberatsa, jokalari talentudunak eta titulu ugariak direla eta. Azken finean, iritziak alda daitezke, zaleen leialtasunak eta eskualdeko harrotasunak eraginda.


In [ ]:
torchaudio.save("audioak/erantzuna.wav", answer.cpu(), sample_rate=16000)